# Notebook 04 — Avaliação Final e Interpretabilidade

## Tech Challenge Fase 1 — Classificação de SRAG (SIVEP-Gripe)

**Objetivo:** Avaliar os modelos treinados no conjunto de **teste** (dados nunca vistos), gerar visualizações completas e interpretar os resultados com técnicas de explicabilidade (SHAP).

> **Importante:** O conjunto de teste é usado APENAS neste notebook, após todas as decisões de modelagem estarem tomadas.

---

## Estrutura do notebook
1. Configuração e carregamento
2. Avaliação de todos os modelos no teste
3. Matrizes de confusão
4. Curvas ROC e Precision-Recall
5. Feature Importance
6. SHAP Values — Explicabilidade
7. Análise crítica e discussão
8. Conclusões finais

## 1. Configuração

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, accuracy_score, recall_score, precision_score,
)

warnings.filterwarnings('ignore')

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.tabular.evaluation import (
    calcular_metricas, comparar_modelos,
    plotar_matriz_confusao, plotar_curvas_roc,
    plotar_curvas_precision_recall, plotar_feature_importance,
    plotar_shap, avaliar_todos_modelos,
)

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
print('Configurado.')

Configurado.


c:\Users\icaro\anaconda3\envs\ia\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Carregamento dos Dados e Modelos

In [ ]:
PROCESSED = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'results' / 'models'

# Dados de teste
X_test = pd.read_parquet(PROCESSED / 'X_test.parquet')
y_test = pd.read_parquet(PROCESSED / 'y_test.parquet').squeeze()

print(f'Conjunto de teste: {X_test.shape[0]:,} amostras, {X_test.shape[1]} features')
print(f'Taxa de óbito no teste: {y_test.mean()*100:.1f}%')

# Modelos treinados
modelos = {}
for arquivo in sorted(MODELS_DIR.glob('*.pkl')):
    modelos[arquivo.stem] = joblib.load(arquivo)
    print(f'Modelo carregado: {arquivo.stem}')

print(f'\nTotal de modelos carregados: {len(modelos)}')

## 3. Avaliação de Todos os Modelos no Conjunto de Teste

### Por que essas métricas?

O dataset é **desbalanceado**: a maioria dos pacientes sobrevive (~75-80% Cura). Nesse contexto:

- **Accuracy** é enganosa — um modelo que previsse sempre "Cura" já teria ~75% de acerto sem nenhum valor clínico
- **F1-score (Óbito)** é a **métrica principal**: equilibra Precision e Recall para a classe clinicamente crítica. Errar um óbito como Cura (falso negativo) tem custo muito maior do que o inverso
- **Recall (Óbito)** é especialmente importante: mede quantos óbitos reais o modelo consegue identificar — a sensibilidade clínica do sistema
- **ROC-AUC** permite comparar modelos independentemente do threshold de decisão, sendo robusta ao desbalanceamento
- **Precision** controla alarmes falsos — relevante para não sobrecarregar a equipe com alertas desnecessários

> Em contexto hospitalar, **maximizar recall** (detectar todos os casos de risco) é preferível a maximizar precision, pois o custo de não identificar um paciente grave é muito maior que o de uma triagem extra.

**Métricas calculadas para cada modelo no conjunto de teste:**
- **Accuracy** — proporção de predições corretas (referência)
- **Precision** — dos preditos como óbito, quantos realmente são
- **Recall** — dos óbitos reais, quantos o modelo detectou *(clinicamente crítico)*
- **F1-score** — média harmônica entre Precision e Recall *(métrica principal)*
- **ROC-AUC** — capacidade discriminativa geral

In [ ]:
resultados = []
for nome, modelo in modelos.items():
    metricas = calcular_metricas(nome, modelo, X_test, y_test)
    resultados.append(metricas)

df_resultados = comparar_modelos(resultados)

In [ ]:
# Heatmap de métricas
fig, ax = plt.subplots(figsize=(10, 4))
metricas_cols = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
df_heat = df_resultados[metricas_cols].astype(float)
sns.heatmap(df_heat, annot=True, fmt='.4f', cmap='RdYlGn', ax=ax,
            vmin=0.5, vmax=1.0, linewidths=0.5)
ax.set_title('Métricas por Modelo — Conjunto de Teste')
plt.tight_layout()
plt.savefig('../results/figures/heatmap_metricas.png', dpi=150)
plt.show()

## 4. Matrizes de Confusão

In [ ]:
n = len(modelos)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
if n == 1:
    axes = [axes]

for ax, (nome, modelo) in zip(axes, modelos.items()):
    y_pred = modelo.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Cura', 'Óbito'],
                yticklabels=['Cura', 'Óbito'])
    ax.set_title(f'{nome}\nAccuracy: {accuracy_score(y_test, y_pred):.3f}')
    ax.set_xlabel('Predição')
    ax.set_ylabel('Real')

plt.suptitle('Matrizes de Confusão — Conjunto de Teste', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/matrizes_confusao.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Curvas ROC

In [ ]:
plotar_curvas_roc(modelos, X_test, y_test)

## 6. Curvas Precision-Recall

Em datasets desbalanceados, as curvas Precision-Recall são mais informativas que ROC porque não são influenciadas pelo grande número de verdadeiros negativos.

In [ ]:
plotar_curvas_precision_recall(modelos, X_test, y_test)

## 7. Feature Importance

Identifica as variáveis com maior impacto nas predições de cada modelo baseado em árvores.

In [ ]:
for nome, modelo in modelos.items():
    if hasattr(modelo, 'feature_importances_'):
        plotar_feature_importance(nome, modelo, list(X_test.columns))

## 8. SHAP Values — Explicabilidade

**SHAP (SHapley Additive exPlanations)** é uma técnica baseada em teoria dos jogos que explica a contribuição de cada feature para uma predição específica.

**Summary Plot:** Mostra:
- Quais features são mais importantes (eixo Y, ordenado por importância média)
- Como cada feature afeta a predição (cor: vermelho = valor alto, azul = valor baixo)
- A magnitude do impacto (eixo X: SHAP value)

In [ ]:
# SHAP para Random Forest
if 'random_forest' in modelos:
    print('Calculando SHAP para Random Forest...')
    plotar_shap('random_forest', modelos['random_forest'], X_test, n_samples=500)

In [ ]:
# SHAP para XGBoost (mais eficiente com TreeExplainer)
if 'xgboost' in modelos:
    print('Calculando SHAP para XGBoost...')
    plotar_shap('xgboost', modelos['xgboost'], X_test, n_samples=500)

In [ ]:
# SHAP Waterfall — explicação de uma predição individual
if 'xgboost' in modelos:
    modelo_xgb = modelos['xgboost']
    X_sample = X_test.sample(200, random_state=42)
    
    explainer = shap.TreeExplainer(modelo_xgb)
    shap_values = explainer(X_sample)
    
    # Selecionar um caso de óbito (y=1) para explicar
    y_sample = y_test.loc[X_sample.index]
    idx_obito = y_sample[y_sample == 1].index[0]
    pos = X_sample.index.get_loc(idx_obito)
    
    print('Explicação individual — caso de Óbito:')
    fig = plt.figure(figsize=(10, 6))
    shap.waterfall_plot(shap_values[pos], max_display=15, show=False)
    plt.title('SHAP Waterfall — Exemplo de Predição de Óbito')
    plt.tight_layout()
    plt.savefig('../results/figures/shap_waterfall_obito.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. Análise Crítica e Discussão

### Desempenho dos Modelos

A tabela comparativa acima resume as métricas de todos os modelos no conjunto de teste. Pontos-chave para análise:

- **XGBoost e Random Forest** tendem a superar os modelos lineares em F1-score e ROC-AUC, confirmando que as relações entre as features clínicas e o desfecho são não-lineares
- **Regressão Logística** serve como baseline válido: se os modelos complexos não superarem significativamente o baseline, isso indica que o ganho de complexidade não se justifica
- O **Recall** merece atenção especial: um modelo com alto F1 mas baixo recall pode estar sendo conservador demais na identificação de óbitos — clinicamente inaceitável
- A **curva Precision-Recall** é mais informativa que a ROC neste contexto desbalanceado, pois não é inflacionada pela grande quantidade de verdadeiros negativos

> **Nota:** os valores numéricos das métricas dependem da execução do pipeline completo com o dataset real (`INFLUD24-26-06-2025.csv`). As análises qualitativas acima descrevem o padrão esperado e o framework de interpretação.

### Interpretação Clínica das Features Mais Importantes

Com base nos resultados de Feature Importance e SHAP, as variáveis com maior impacto preditivo tendem a ser:

| Feature | Interpretação Clínica |
|---------|----------------------|
| `UTI` | Pacientes em UTI têm risco muito maior de óbito — indica gravidade já na admissão |
| `SUPORT_VEN` | Ventilação invasiva sinaliza insuficiência respiratória grave |
| `NU_IDADE_N` | Idade avançada é fator de risco consolidado na literatura de SRAG |
| `SATURACAO` | Saturação baixa (valor 1) é marcador direto de hipoxemia |
| `CARDIOPATI` | Comorbidade cardiovascular eleva mortalidade em doenças respiratórias |
| `DISPNEIA` | Sintoma de gravidade respiratória presente na admissão |
| `DIABETES` | Comorbidade associada a desfechos piores em infecções respiratórias |

O SHAP Waterfall acima mostra como cada feature empurra a predição para Cura ou Óbito em um caso individual — isso é fundamental para que o clínico entenda *por que* o modelo sinalizou risco em um paciente específico.

### O modelo pode ser usado na prática?

**Potenciais aplicações como ferramenta de apoio:**
- **Triagem no pronto-socorro:** sinalizar automaticamente pacientes de alto risco para avaliação prioritária
- **Priorização de UTI:** quando a demanda excede a oferta, o score de risco auxilia na tomada de decisão
- **Vigilância epidemiológica:** monitorar perfis populacionais de risco sem tomada de decisão individual

**Limitações que impedem uso autônomo:**
- Treinado em dados de 2024 — pode não generalizar para novos agentes virais ou variantes
- Não captura informações qualitativas do exame físico nem a evolução nas primeiras horas
- Features faltantes degradam a predição silenciosamente
- O threshold de 0.5 não foi otimizado para o custo assimétrico dos erros clínicos

### O médico DEVE ter a palavra final

> Um sistema de IA para apoio à decisão clínica deve ser tratado como um **segundo par de olhos**, não como substituto do julgamento médico. O fluxo correto é: **modelo sinaliza risco → médico avalia o contexto completo → médico decide.**
>
> O profissional de saúde considera histórico completo, exame físico, evolução clínica e fatores contextuais que nenhum modelo de dados estruturados captura. A responsabilidade pelo diagnóstico e pela conduta é sempre do médico.

## 10. Conclusões Finais

In [ ]:
# Tabela final de resultados
print('=' * 70)
print('RESULTADOS FINAIS — CONJUNTO DE TESTE')
print('=' * 70)
print(df_resultados.to_string())
print('=' * 70)

melhor_modelo = df_resultados['f1'].idxmax()
print(f'\nMelhor modelo (F1-score): {melhor_modelo}')
print(f'F1-score:  {df_resultados.loc[melhor_modelo, "f1"]:.4f}')
print(f'ROC-AUC:   {df_resultados.loc[melhor_modelo, "roc_auc"]:.4f}')
print(f'Recall:    {df_resultados.loc[melhor_modelo, "recall"]:.4f}')

In [ ]:
print('\nFiguras geradas em results/figures/:')
for f in sorted((ROOT / 'results' / 'figures').glob('*.png')):
    print(f'  {f.name}')